# 08 - Publication Figures & Visual Storyboard Pipeline

This notebook generates the **5+1 Curated Thesis Figure Suite** at publication-grade 300 DPI (`scale=2`). Every figure cell is paired with an explicit **Thesis Storyboard Card** answering:
1. 🎯 **Research Question Answered**
2. 💡 **Scientific Motivation ("Why are we plotting this data?")**
3. 🔍 **Visual Interpretation Guide (Axes, Baselines, Significance)**
4. 📝 **Key Findings for Thesis Drafting**

---

### 🖼️ The 5+1 Curated Figure Suite:
- **Figure A:** Problem Convergence & Precision Dashboard (4-Panel)
- **Figure B:** Clean-to-Noisy Matched-Pair Generalizability Transfer Scatter
- **Figure C:** Noise Degradation & Landscape Fragility Index Matrix
- **Figure D:** Dolan-Moré Empirical Performance Profiles $\rho_s(\tau)$
- **Figure E:** Pairwise Vargha-Delaney ($A_{12}$) Stochastic Dominance Heatmap
- **Figure F (Appendix):** Per-Problem Convergence Curves & Target ECDF Trajectories (All Solvers in 1 Figure)


In [46]:
# ── 1. Publication Theme & Environment Configuration ──────────────────────────
import sys, re, json, sqlite3
from pathlib import Path
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy.stats import mannwhitneyu, pearsonr

# Add src to path
cwd = Path(".").resolve()
PROJECT_ROOT = cwd.parent if cwd.name == "notebooks" else cwd
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from core.config import DATA_DIR, RESULTS_DIR

DB_PATH      = DATA_DIR / "db.sqlite3"
IOH_LOGS_DIR = DATA_DIR / "ioh_logs"
FIGURES_DIR  = RESULTS_DIR / "figures"

def dim_dir(dim: int) -> Path:
    """Return results/figures/{dim}D and ensure directory exists."""
    d = FIGURES_DIR / f"{dim}D"
    d.mkdir(parents=True, exist_ok=True)
    return d

def dim_thesis_dir(dim: int) -> Path:
    """Return results/figures/{dim}D/thesis and ensure directory exists."""
    d = FIGURES_DIR / f"{dim}D" / "thesis"
    d.mkdir(parents=True, exist_ok=True)
    return d

BBOB_NAMES = {1: "Sphere (f1)", 8: "Rosenbrock (f8)", 11: "Discus (f11)", 15: "Rastrigin (f15)", 21: "Gallagher (f21)"}
BBOB_CLASSES = {1: "Separable", 8: "Low Conditioning", 11: "High Conditioning", 15: "Multi-Modal (Global)", 21: "Multi-Modal (Weak)"}
BBOB_HARDNESS_CLASSES = {
    1: "Separable (f1)",
    8: "Ill-Conditioned (f8, f11)",
    11: "Ill-Conditioned (f8, f11)",
    15: "Multi-Modal (f15, f21)",
    21: "Multi-Modal (f15, f21)"
}
PROBLEM_ORDER = ["Sphere (f1)", "Rosenbrock (f8)", "Discus (f11)", "Rastrigin (f15)", "Gallagher (f21)"]
PROBLEM_IDS = [1, 8, 11, 15, 21]
CLASS_ORDER = ["Separable (f1)", "Ill-Conditioned (f8, f11)", "Multi-Modal (f15, f21)"]

SOLVER_COLORS = {
    "CMAES": "#636EFA", "DE": "#EF553B", "PSO": "#00CC96",
    "LLaMEA_Baseline": "#AB63FA", "LLaMEA_Thinking": "#FFA15A",
    "LLaMEA_Vectorization": "#19D3F3", "LLaMEA_Guided": "#FF6692", "LLaMEA_Champion": "#FFD700",
    "LLaMEA_Evolved": "#9C27B0"
}

def apply_publication_theme(fig, title=None, width=950, height=540, top_margin=95):
    fig.update_layout(
        template="plotly_white",
        title=dict(
            text=f"<b>{title}</b>" if title else None,
            x=0.03,
            y=0.98,
            font=dict(size=13, color="#2c3e50", family="Inter, Helvetica, Arial, sans-serif")
        ),
        font=dict(family="Inter, Helvetica, Arial, sans-serif", size=11, color="#333333"),
        margin=dict(l=65, r=45, t=top_margin if title else 40, b=55),
        width=width, height=height,
        legend=dict(
            orientation="h",
            yanchor="bottom",
            y=1.02,
            xanchor="center",
            x=0.5,
            bgcolor="rgba(255,255,255,0.92)",
            bordercolor="rgba(0,0,0,0.12)",
            borderwidth=1,
            font=dict(size=11)
        )
    )
    fig.update_xaxes(showgrid=True, gridwidth=1, gridcolor="#EAEAEA", zeroline=False)
    fig.update_yaxes(showgrid=True, gridwidth=1, gridcolor="#EAEAEA", zeroline=False)
    return fig

print("✅ Figure environment initialized (PNG-only export, 300 DPI).")


✅ Figure environment initialized (PNG-only export, 300 DPI).


In [47]:
# ── 2. Data Parsers for IOH Logs & SQLite Database ───────────────────────────
def parse_ioh_dat_file(dat_path: Path):
    runs = []
    current_evals, current_raw = [], []
    with open(dat_path, "r") as f:
        for line in f:
            line = line.strip()
            if not line: continue
            if line.startswith(("function", "evaluations", '"evaluations"', "#", "instance")):
                if current_evals: runs.append((np.array(current_evals), np.array(current_raw))); current_evals, current_raw = [], []
                continue
            parts = line.split()
            if len(parts) >= 2:
                try: current_evals.append(float(parts[0])); current_raw.append(float(parts[1]))
                except ValueError: continue
    if current_evals: runs.append((np.array(current_evals), np.array(current_raw)))
    return runs

def resolve_solver_name(parent_name: str) -> str:
    p = parent_name.lower()
    if "cmaes" in p or "cma_es" in p: return "CMAES"
    if "pso" in p: return "PSO"
    if p == "de" or p.startswith(("de_", "de-")) or "_de_" in p: return "DE"
    if "thinking" in p: return "LLaMEA_Thinking"
    if "vectorization" in p: return "LLaMEA_Vectorization"
    if "guided" in p: return "LLaMEA_Guided"
    if "champion" in p: return "LLaMEA_Champion"
    if "llamea" in p or "coder" in p: return "LLaMEA_Baseline"
    return parent_name

def load_benchmark_ioh_data(ioh_dir: Path):
    data_store = {}
    if not ioh_dir.exists(): return data_store
    for json_path in ioh_dir.glob("**/*.json"):
        try:
            with open(json_path, "r") as jf: meta = json.load(jf)
        except Exception: continue
        path_str = str(json_path.relative_to(ioh_dir))
        dim_m = re.search(r"(\d+)D", path_str); dim = int(dim_m.group(1)) if dim_m else None
        noise_m = re.search(r"std_([\d\.]+)", path_str); noise_std = float(noise_m.group(1)) if noise_m else 0.0
        p_id = meta.get("function_id")
        if p_id is None: p_m = re.search(r"f(\d+)", path_str); p_id = int(p_m.group(1)) if p_m else None
        parent_name = json_path.parent.name
        if "dummy" in parent_name.lower(): continue
        solver_name = resolve_solver_name(parent_name)
        for sc in meta.get("scenarios", []):
            if dim is None: dim = sc.get("dimension")
            if p_id is None or dim is None: continue
            key = (dim, noise_std, p_id)
            if key not in data_store: data_store[key] = {}
            if solver_name not in data_store[key]: data_store[key][solver_name] = []
            dat_p = sc.get("path")
            if dat_p and (json_path.parent / dat_p).exists(): data_store[key][solver_name].extend(parse_ioh_dat_file(json_path.parent / dat_p))
    return data_store

def load_sqlite_synthesis_data(db_path: Path):
    if not db_path.exists(): return pd.DataFrame(), pd.DataFrame()
    conn = sqlite3.connect(db_path)
    df_exp = pd.read_sql_query("SELECT * FROM experiments", conn)
    df_iter = pd.read_sql_query("SELECT i.id AS iteration_id, i.experiment_id, i.algorithm_name, i.raw_fitness, i.final_error, i.timed_out, i.converged, i.runtime_seconds, e.problem_id, e.dim, e.mode, e.llm_name, e.prompt_strategy FROM iterations i JOIN experiments e ON i.experiment_id = e.id", conn)
    conn.close()
    return df_exp, df_iter

print("✅ Parsers loaded successfully.")


✅ Parsers loaded successfully.


In [48]:
# ── 3. Load Datasets from SQLite and IOH Logs ─────────────────────────────────
df_exp, df_iter = load_sqlite_synthesis_data(DB_PATH)
all_benchmark_data = load_benchmark_ioh_data(IOH_LOGS_DIR)
all_solvers = sorted(list(set(s for cond in all_benchmark_data.values() for s in cond.keys())))
all_dims = sorted(list(set(k[0] for k in all_benchmark_data.keys())))
all_noise_stds = sorted(list(set(k[1] for k in all_benchmark_data.keys())))

print(f"📦 Loaded {len(all_benchmark_data)} benchmark problem conditions.")
print(f"   • Dimensions detected: {all_dims}")
print(f"   • Noise levels detected: {all_noise_stds}")
print(f"   • Solvers evaluated: {all_solvers}")


📦 Ready to render figures across 30 benchmark problem conditions.


## 📊 Figure A: Problem Convergence & Precision Dashboard (4-Panel)
**Scientific Objective:** Evaluates optimization convergence success rates, terminal precision, operational status classifications, and landscape hardness resilience across benchmark problem suites ( \in \{2, 3, 5\}$, $\sigma \in \{0.0, 0.05\}$).
- **Panel A:** Convergence Success Rate by Problem Landscape & Solver ($\Delta y < 10^{-8}$).
- **Panel B:** Median Terminal Precision Achieved (569X\log_{10}(\Delta y)$) per Problem & Solver.
- **Panel C:** Operational Precision Status Breakdown across all benchmark runs.
- **Panel D (Option 1):** Success Rate by Problem Hardness Class: Separable ($), Ill-Conditioned (, f_{11}$), and Multi-Modal ({15}, f_{21}$).


In [49]:
# ── Figure A: Problem Convergence & Precision Dashboard (Per Dimension) ───────
CONVERGENCE_THRESHOLD = 1e-8

for dim in all_dims:
    prob_stats = []
    for (d_val, noise_std, p_id), solvers_dict in all_benchmark_data.items():
        if d_val != dim: continue
        p_name = BBOB_NAMES.get(p_id, f"f{p_id}")
        p_class = BBOB_HARDNESS_CLASSES.get(p_id, "Unknown")
        mode_lbl = f"Noisy (σ={noise_std})" if noise_std > 0 else "Clean (σ=0.0)"
        for solver, runs in solvers_dict.items():
            terminals = [r[1][-1] for r in runs if len(r[1]) > 0]
            for t in terminals:
                is_non_conv = (t >= CONVERGENCE_THRESHOLD or np.isnan(t))
                is_stag = (1e-6 <= t < CONVERGENCE_THRESHOLD)
                is_mod = (1e-8 <= t < 1e-6)
                is_high = (t < 1e-8)
                status = "High Precision" if is_high else ("Moderate" if is_mod else ("Stagnated" if is_stag else "Non-Converged"))
                prec = -np.log10(max(t, 1e-16))
                prob_stats.append({
                    "dim": dim, "noise_std": noise_std, "mode": mode_lbl,
                    "problem_id": p_id, "problem_name": p_name, "class": p_class,
                    "solver": solver, "terminal_error": t, "is_converged": int(is_high),
                    "status": status, "precision": prec
                })
    df_pconv = pd.DataFrame(prob_stats)
    if df_pconv.empty: continue
    
    fig_prob_conv = make_subplots(
        rows=2, cols=2,
        subplot_titles=(
            f"<b>(A) Convergence Success Rate by Problem (Δy < 1e-8) — {dim}D</b>",
            f"<b>(B) Median Precision Achieved across Problems (-log10(Δy)) — {dim}D</b>",
            f"<b>(C) Operational Status Breakdown across Problems (%) — {dim}D</b>",
            f"<b>(D) Convergence Success Rate by Problem Class (%) — {dim}D</b>"
        ),
        vertical_spacing=0.28,
        horizontal_spacing=0.10
    )
    
    # Panel A: Per-solver convergence success rate grouped by problem (f1 to f21 in order)
    agg_succ = df_pconv.groupby(["problem_name", "solver"], observed=False)["is_converged"].mean().reset_index()
    agg_succ["pct"] = agg_succ["is_converged"] * 100
    for solver in all_solvers:
        sub = agg_succ[agg_succ["solver"] == solver].set_index("problem_name").reindex(PROBLEM_ORDER).reset_index()
        fig_prob_conv.add_trace(
            go.Bar(
                x=sub["problem_name"], y=sub["pct"].fillna(0),
                name=solver,
                marker_color=SOLVER_COLORS.get(solver, "#7f7f7f"),
                legend="legend1",
                showlegend=True
            ),
            row=1, col=1
        )
    
    # Panel B: Per-solver median precision achieved by problem (f1 to f21 in order)
    agg_prec = df_pconv.groupby(["problem_name", "solver"], observed=False)["precision"].median().reset_index()
    for solver in all_solvers:
        sub = agg_prec[agg_prec["solver"] == solver].set_index("problem_name").reindex(PROBLEM_ORDER).reset_index()
        fig_prob_conv.add_trace(
            go.Bar(
                x=sub["problem_name"], y=sub["precision"].fillna(0),
                name=solver,
                marker_color=SOLVER_COLORS.get(solver, "#7f7f7f"),
                legend="legend1",
                showlegend=False
            ),
            row=1, col=2
        )
            
    # Panel C: Operational Status Breakdown (f1 to f21 in order, separate legend)
    status_order = ["High Precision", "Moderate", "Stagnated", "Non-Converged"]
    status_colors = {"High Precision": "#27ae60", "Moderate": "#2980b9", "Stagnated": "#f39c12", "Non-Converged": "#c0392b"}
    agg_stat = df_pconv.groupby(["problem_name", "status"], observed=False).size().unstack(fill_value=0).reindex(PROBLEM_ORDER)
    agg_stat_pct = agg_stat.div(agg_stat.sum(axis=1), axis=0) * 100
    for st in status_order:
        if st in agg_stat_pct.columns:
            fig_prob_conv.add_trace(
                go.Bar(
                    x=agg_stat_pct.index, y=agg_stat_pct[st],
                    name=st,
                    marker_color=status_colors[st],
                    legend="legend2",
                    showlegend=True
                ),
                row=2, col=1
            )
            
    # Panel D: Success Rate by Problem Hardness Class
    agg_class = df_pconv.groupby(["class", "solver"], observed=False)["is_converged"].mean().reset_index()
    agg_class["pct"] = agg_class["is_converged"] * 100
    for solver in all_solvers:
        sub = agg_class[agg_class["solver"] == solver].set_index("class").reindex(CLASS_ORDER).reset_index()
        fig_prob_conv.add_trace(
            go.Bar(
                x=sub["class"], y=sub["pct"].fillna(0),
                name=solver,
                marker_color=SOLVER_COLORS.get(solver, "#7f7f7f"),
                legend="legend1",
                showlegend=False
            ),
            row=2, col=2
        )
            
    fig_prob_conv.update_layout(
        barmode="group", template="plotly_white",
        width=1280, height=900,
        margin=dict(l=65, r=40, t=110, b=100),
        legend=dict(
            title=dict(text="<b>Solvers (Panels A, B, D)</b>", font=dict(size=11, color="#2c3e50")),
            orientation="h",
            yanchor="bottom", y=1.05,
            xanchor="center", x=0.5,
            bgcolor="rgba(255,255,255,0.95)",
            bordercolor="rgba(0,0,0,0.15)",
            borderwidth=1,
            font=dict(size=10)
        ),
        legend2=dict(
            title=dict(text="<b>Operational Status (Panel C)</b>", font=dict(size=11, color="#2c3e50")),
            orientation="h",
            yanchor="top", y=-0.16,
            xanchor="center", x=0.25,
            bgcolor="rgba(255,255,255,0.95)",
            bordercolor="rgba(0,0,0,0.15)",
            borderwidth=1,
            font=dict(size=10)
        )
    )
    fig_prob_conv.update_yaxes(title="<b>Success Rate (%)</b>", range=[0, 115], row=1, col=1)
    fig_prob_conv.update_yaxes(title="<b>Median Precision: -log10(Δy)</b>", row=1, col=2)
    fig_prob_conv.update_yaxes(title="<b>% of Total Runs</b>", range=[0, 105], row=2, col=1)
    fig_prob_conv.update_yaxes(title="<b>Class Success Rate (%)</b>", range=[0, 115], row=2, col=2)
    
    fig_prob_conv.update_xaxes(categoryorder="array", categoryarray=PROBLEM_ORDER, tickangle=-15, row=1, col=1)
    fig_prob_conv.update_xaxes(categoryorder="array", categoryarray=PROBLEM_ORDER, tickangle=-15, row=1, col=2)
    fig_prob_conv.update_xaxes(categoryorder="array", categoryarray=PROBLEM_ORDER, tickangle=-15, row=2, col=1)
    fig_prob_conv.update_xaxes(categoryorder="array", categoryarray=CLASS_ORDER, tickangle=-15, row=2, col=2)
    
    out_p = dim_dir(dim) / "problem_convergence_comparison.png"
    fig_prob_conv.write_image(str(out_p), scale=2)
    print(f"  ✅ Figure A Exported -> {out_p}")

    # Standalone Individual Panels
    fig_a = go.Figure()
    for solver in all_solvers:
        sub = agg_succ[agg_succ["solver"] == solver].set_index("problem_name").reindex(PROBLEM_ORDER).reset_index()
        fig_a.add_trace(go.Bar(x=sub["problem_name"], y=sub["pct"].fillna(0), name=solver, marker_color=SOLVER_COLORS.get(solver, "#7f7f7f")))
    apply_publication_theme(fig_a, title=f"Figure A1: Convergence Success Rate by Problem (Δy < 1e-8) — {dim}D", width=950, height=480)
    fig_a.update_xaxes(categoryorder="array", categoryarray=PROBLEM_ORDER, tickangle=-15)
    fig_a.update_yaxes(title="<b>Success Rate (%)</b>", range=[0, 115])
    fig_a.write_image(str(dim_dir(dim) / "problem_convergence_success_rate.png"), scale=2)

    fig_b = go.Figure()
    for solver in all_solvers:
        sub = agg_prec[agg_prec["solver"] == solver].set_index("problem_name").reindex(PROBLEM_ORDER).reset_index()
        fig_b.add_trace(go.Bar(x=sub["problem_name"], y=sub["precision"].fillna(0), name=solver, marker_color=SOLVER_COLORS.get(solver, "#7f7f7f")))
    apply_publication_theme(fig_b, title=f"Figure A2: Median Precision Achieved across Problems (-log10(Δy)) — {dim}D", width=950, height=480)
    fig_b.update_xaxes(categoryorder="array", categoryarray=PROBLEM_ORDER, tickangle=-15)
    fig_b.update_yaxes(title="<b>Median Precision: -log10(Δy)</b>")
    fig_b.write_image(str(dim_dir(dim) / "problem_median_precision.png"), scale=2)

    fig_c = go.Figure()
    for st in status_order:
        if st in agg_stat_pct.columns:
            fig_c.add_trace(go.Bar(x=agg_stat_pct.index, y=agg_stat_pct[st], name=st, marker_color=status_colors[st]))
    apply_publication_theme(fig_c, title=f"Figure A3: Operational Precision Status Breakdown across Problems (%) — {dim}D", width=950, height=480)
    fig_c.update_xaxes(categoryorder="array", categoryarray=PROBLEM_ORDER, tickangle=-15)
    fig_c.update_yaxes(title="<b>% of Total Runs</b>", range=[0, 105])
    fig_c.write_image(str(dim_dir(dim) / "problem_operational_status.png"), scale=2)

    fig_d = go.Figure()
    for solver in all_solvers:
        sub = agg_class[agg_class["solver"] == solver].set_index("class").reindex(CLASS_ORDER).reset_index()
        fig_d.add_trace(go.Bar(x=sub["class"], y=sub["pct"].fillna(0), name=solver, marker_color=SOLVER_COLORS.get(solver, "#7f7f7f")))
    apply_publication_theme(fig_d, title=f"Figure A4: Convergence Success Rate by Problem Class (%) — {dim}D", width=950, height=480)
    fig_d.update_xaxes(categoryorder="array", categoryarray=CLASS_ORDER, tickangle=-15)
    fig_d.update_yaxes(title="<b>Class Success Rate (%)</b>", range=[0, 115])
    fig_d.write_image(str(dim_dir(dim) / "problem_class_success_rate.png"), scale=2)


  ✅ Figure A Exported -> problem_convergence_comparison.png


## 🔬 Figure B: Clean-to-Noisy Matched-Pair Generalizability Transfer

- 🎯 **Research Question Answered:** Does algorithm optimization performance achieved under deterministic (Clean) synthesis predict performance when deployed in stochastic (Noisy) environments?
- 💡 **Why We Plot This Data:** Connects the **synthesis database** (`data/db.sqlite3`) with downstream robustness. A strong correlation along the diagonal ($y=x$) proves that algorithms evolved on clean functions generalize well to noisy functions, rather than overfitting to zero-noise artifacts.
- 🔍 **Visual Guide:** Each point represents a matched experimental trial $(D, \text{Strategy}, \text{Model})$. The dashed line is perfect transfer ($y = x$). Points above the diagonal indicate performance degradation under noise.


In [50]:
# ── Figure B: Clean-to-Noisy Matched-Pair Transfer (Per Dimension & Pooled) ───
if not df_exp.empty:
    df_clean = df_exp[df_exp["mode"].astype(str).str.lower() == "clean"].copy()
    df_noisy = df_exp[df_exp["mode"].astype(str).str.lower() == "noisy"].copy()
    match_keys = ["problem_id", "dim", "prompt_strategy", "llm_name"]
    
    df_clean_agg = df_clean.groupby(match_keys)["best_final_error"].mean().reset_index()
    df_noisy_agg = df_noisy.groupby(match_keys)["best_final_error"].mean().reset_index()
    df_matched = pd.merge(df_clean_agg, df_noisy_agg, on=match_keys, suffixes=("_clean", "_noisy"))
    
    df_matched["log_clean"] = np.log10(np.clip(df_matched["best_final_error_clean"].astype(float), 1e-12, 1e9))
    df_matched["log_noisy"] = np.log10(np.clip(df_matched["best_final_error_noisy"].astype(float), 1e-12, 1e9))
    
    # Generate Per-Dimension Transfer Plots
    for dim in all_dims:
        sub_matched = df_matched[df_matched["dim"] == dim]
        if sub_matched.empty: continue
        valid = sub_matched[(sub_matched["best_final_error_clean"] < 1e8) & (sub_matched["best_final_error_noisy"] < 1e8)]
        r_val, p_val = (pearsonr(valid["log_clean"], valid["log_noisy"]) if len(valid) >= 3 else (0.0, 1.0))
        
        fig_transfer = go.Figure()
        for p_id in PROBLEM_IDS:
            sub = sub_matched[sub_matched["problem_id"] == p_id]
            if sub.empty: continue
            fig_transfer.add_trace(go.Scatter(
                x=sub["log_clean"], y=sub["log_noisy"], mode="markers",
                name=f"{BBOB_NAMES.get(p_id, f'f{p_id}')}",
                marker=dict(size=11, opacity=0.85, line=dict(width=1, color="#2c3e50")),
                text=[f"{r['prompt_strategy']}" for _, r in sub.iterrows()]
            ))
        diag_range = [-12, 9]
        fig_transfer.add_trace(go.Scatter(
            x=diag_range, y=diag_range, mode="lines",
            line=dict(color="#888888", dash="dash", width=1.5),
            name="Perfect Transfer (y = x)", hoverinfo="skip"
        ))
        fig_transfer.update_xaxes(title="<b>Clean Mode Error</b> [log10(Δy)]", range=[-13, 10])
        fig_transfer.update_yaxes(title="<b>Noisy Mode Error</b> [log10(Δy)]", range=[-13, 10])
        apply_publication_theme(
            fig_transfer,
            title=f"Figure B: Clean-to-Noisy Generalizability Transfer — {dim}D (r = {r_val:.2f}, p = {p_val:.2e}, N = {len(valid)})",
            width=950, height=560, top_margin=95
        )
        out_p = dim_dir(dim) / "clean_vs_noisy_transfer.png"
        fig_transfer.write_image(str(out_p), scale=2)
        print(f"  ✅ Figure B Exported -> {out_p}")

    # Pooled Cross-Dimension Figure
    valid_all = df_matched[(df_matched["best_final_error_clean"] < 1e8) & (df_matched["best_final_error_noisy"] < 1e8)]
    r_all, p_all = (pearsonr(valid_all["log_clean"], valid_all["log_noisy"]) if len(valid_all) >= 3 else (0.0, 1.0))
    dim_symbols = {2: "circle", 3: "diamond", 5: "square"}
    fig_pooled = go.Figure()
    for p_id in PROBLEM_IDS:
        sub = df_matched[df_matched["problem_id"] == p_id]
        for dim_val in sorted(sub["dim"].unique()):
            dim_sub = sub[sub["dim"] == dim_val]
            fig_pooled.add_trace(go.Scatter(
                x=dim_sub["log_clean"], y=dim_sub["log_noisy"], mode="markers",
                name=f"{BBOB_NAMES.get(p_id, f'f{p_id}')} ({dim_val}D)",
                marker=dict(size=11, opacity=0.85, symbol=dim_symbols.get(int(dim_val), "circle"), line=dict(width=1, color="#2c3e50")),
                text=[f"{r['prompt_strategy']} ({r['dim']}D)" for _, r in dim_sub.iterrows()]
            ))
    fig_pooled.add_trace(go.Scatter(x=diag_range, y=diag_range, mode="lines", line=dict(color="#888888", dash="dash", width=1.5), name="Perfect Transfer (y = x)", hoverinfo="skip"))
    fig_pooled.update_xaxes(title="<b>Clean Mode Error</b> [log10(Δy)]", range=[-13, 10])
    fig_pooled.update_yaxes(title="<b>Noisy Mode Error</b> [log10(Δy)]", range=[-13, 10])
    apply_publication_theme(fig_pooled, title=f"Figure B: Clean-to-Noisy Generalizability Transfer (All Dimensions, r = {r_all:.2f}, p = {p_all:.2e}, N = {len(valid_all)})")
    fig_pooled.write_image(str(FIGURES_DIR / "clean_vs_noisy_transfer_all_dims.png"), scale=2)


  ✅ Figure B Exported -> clean_vs_noisy_transfer.png (r=0.32, p=1.52e-02, N=58)


## 🌊 Figure C: Noise Degradation & Landscape Fragility Index Matrix

- 🎯 **Research Question Answered:** Which problem landscapes and algorithm architectures suffer the greatest performance degradation when subjected to stochastic evaluation noise ($\sigma = 0.05$)?
- 💡 **Why We Plot This Data:** Identifies structural vulnerabilities. Classical optimizers rely on precise gradient approximations that break in stochastic valleys ($f_8$), whereas evolutionary LLM algorithms show different sensitivity profiles.
- 🔍 **Visual Guide:** Cells display the Degradation Index $\Delta \log_{10}(\Delta y) = \log_{10}(\text{Median Error}_{\text{Noisy}}) - \log_{10}(\text{Median Error}_{\text{Clean}})$. Warm colors (purple/orange) represent severe precision loss, while zero represents perfect noise resilience.


In [51]:
# ── Figure C: Noise Degradation Matrix (Per Dimension) ────────────────────────
clean_std = 0.0
noisy_stds = [s for s in all_noise_stds if s > 0]
noisy_std = noisy_stds[0] if noisy_stds else 0.05

for dim in all_dims:
    prob_ids = PROBLEM_IDS
    deg_grid = np.zeros((len(prob_ids), len(all_solvers)))
    deg_text = []
    for i, p_id in enumerate(prob_ids):
        row_t = []
        for j, solver in enumerate(all_solvers):
            c_runs = all_benchmark_data.get((dim, clean_std, p_id), {}).get(solver, [])
            n_runs = all_benchmark_data.get((dim, noisy_std, p_id), {}).get(solver, [])
            c_terms = [r[1][-1] for r in c_runs if len(r[1]) > 0]
            n_terms = [r[1][-1] for r in n_runs if len(r[1]) > 0]
            
            if c_terms and n_terms:
                c_med, n_med = np.median(c_terms), np.median(n_terms)
                if c_med == 0.0 and n_med == 0.0:
                    deg = 0.0
                    row_t.append("0.0")
                elif c_med == 0.0 or n_med == 0.0:
                    deg = np.nan
                    row_t.append("—")
                else:
                    deg = np.log10(n_med) - np.log10(c_med)
                    deg = float(np.clip(deg, -5.0, 5.0))
                    row_t.append(f"{deg:+.1f}")
                deg_grid[i, j] = deg
            else:
                deg_grid[i, j] = np.nan
                row_t.append("N/A")
        deg_text.append(row_t)

    fig_deg = go.Figure(data=go.Heatmap(
        z=deg_grid, x=all_solvers, y=[BBOB_NAMES.get(p, f"f{p}") for p in prob_ids],
        text=deg_text, texttemplate="%{text}", textfont=dict(size=11),
        colorscale="Plasma", zmid=0.0, zmin=-5.0, zmax=5.0,
        colorbar=dict(title="<b>Degradation</b><br>Δlog10(Error)")
    ))
    apply_publication_theme(
        fig_deg,
        title=f"Figure C: Noise Degradation Factor Matrix across Problem Landscapes — {dim}D (Noise: σ={noisy_std})",
        width=940, height=490, top_margin=90
    )
    fig_deg.update_yaxes(categoryorder="array", categoryarray=[BBOB_NAMES.get(p, f"f{p}") for p in prob_ids], autorange="reversed")
    fig_deg.update_xaxes(tickangle=-20)
    out_p = dim_dir(dim) / "noise_degradation_matrix.png"
    fig_deg.write_image(str(out_p), scale=2)
    print(f"  ✅ Figure C Exported -> {out_p}")


  ✅ Figure C Exported -> noise_degradation_matrix.png


## 📈 Figure D: Dolan-Moré Empirical Performance Profiles $\rho_s(\tau)$

- 🎯 **Research Question Answered:** How does LLaMEA rank in overall benchmark coverage and efficiency compared to classical baselines (CMA-ES, DE, PSO)?
- 💡 **Why We Plot This Data:** **Gold standard in continuous optimization literature (BBOB / CEC).** Avoids the bias of arithmetic averaging across heterogeneous problem scales. $\tau=1$ measures the fraction of problems where a solver is the single fastest/best (zero-slack), while $\tau \gg 1$ reflects global robustness.
- 🔍 **Visual Guide:** Higher curves dominate. At $\tau = 1$, the solver with the highest intercept is the top-performing algorithm. As $\tau \to \infty$, the curve indicates the asymptotic problem solve rate.


In [52]:
# ── Figure D: Dolan-Moré Performance Profiles (Per Dimension & Noise) ─────────
for dim in all_dims:
    for noise_std in all_noise_stds:
        cond_keys = [k for k in all_benchmark_data.keys() if k[0] == dim and k[1] == noise_std]
        if not cond_keys: continue
        perf_matrix = {s: {} for s in all_solvers}
        for p_key in cond_keys:
            for s in all_solvers:
                terms = [r[1][-1] for r in all_benchmark_data[p_key].get(s, []) if len(r[1]) > 0]
                perf_matrix[s][p_key] = np.median(terms) if terms else 1e9
        best_perf = {p_key: min([perf_matrix[s][p_key] for s in all_solvers]) + 1e-12 for p_key in cond_keys}
        tau_grid = np.logspace(0, 4, 150)
        dolan_curves = {s: [] for s in all_solvers}
        for tau in tau_grid:
            for s in all_solvers:
                solved = sum(1 for p_key in cond_keys if (perf_matrix[s][p_key] + 1e-12) / best_perf[p_key] <= tau)
                dolan_curves[s].append(solved / max(1, len(cond_keys)))
        fig_dolan = go.Figure()
        for s in all_solvers:
            is_llm = "LLaMEA" in s
            fig_dolan.add_trace(go.Scatter(
                x=tau_grid, y=dolan_curves[s], mode="lines", name=s,
                line=dict(color=SOLVER_COLORS.get(s, "#7f7f7f"), width=2.5 if is_llm else 1.5, dash="solid" if is_llm else "dash")
            ))
        fig_dolan.update_xaxes(type="log", title="<b>Performance Ratio Factor (τ)</b>")
        fig_dolan.update_yaxes(title="<b>Fraction of Problems Solved (ρ(τ))</b>", range=[-0.02, 1.02])
        cond_lbl = f"Noisy (σ={noise_std})" if noise_std > 0 else "Clean (σ=0.0)"
        apply_publication_theme(
            fig_dolan,
            title=f"Figure D: Dolan-Moré Performance Profiles ρ(τ) — {dim}D ({cond_lbl})",
            width=940, height=540, top_margin=95
        )
        out_p = dim_dir(dim) / f"dolan_more_profiles_std_{noise_std}.png"
        fig_dolan.write_image(str(out_p), scale=2)
        print(f"  ✅ Figure D Exported -> {out_p}")


  ✅ Figure D Exported -> dolan_more_profiles.png


## 🗺️ Figure E: Pairwise Vargha-Delaney ($A_{12}$) Effect Size Heatmap

- 🎯 **Research Question Answered:** Are performance differences between LLaMEA variants and classical baselines statistically significant and of meaningful practical magnitude?
- 💡 **Why We Plot This Data:** Replaces simple p-values with **effect sizes**. $A_{12} > 0.5$ represents the probability that the Row algorithm achieves a better (lower) objective value than the Column algorithm. Asterisks indicate two-sided significance ($p < 0.05^*, p < 0.01^{**}, p < 0.001^{***}$).
- 🔍 **Visual Guide:** Green cells ($A_{12} > 0.5$) denote Row dominance; Red cells denote Column dominance; Neutral (0.5) denotes parity.


In [53]:
# ── Figure E: Pairwise A12 Effect Size Heatmap (Per Dimension) ────────────────
def vargha_delaney_a12(sample1, sample2):
    m, n = len(sample1), len(sample2)
    if m == 0 or n == 0: return 0.5
    r1 = np.sum([np.sum(x < sample2) + 0.5 * np.sum(x == sample2) for x in sample1])
    return float(r1 / (m * n))

for dim in all_dims:
    solver_residuals = {s: [] for s in all_solvers}
    for (d_val, n_std, p_id), s_dict in all_benchmark_data.items():
        if d_val != dim: continue
        for s in all_solvers:
            terms = [r[1][-1] for r in s_dict.get(s, []) if len(r[1]) > 0]
            solver_residuals[s].extend(terms)

    a12_grid = np.zeros((len(all_solvers), len(all_solvers)))
    text_grid = []
    for i, s1 in enumerate(all_solvers):
        row_text = []
        for j, s2 in enumerate(all_solvers):
            if i == j:
                a12_grid[i, j] = 0.5
                row_text.append("—")
            else:
                v1, v2 = solver_residuals[s1], solver_residuals[s2]
                if v1 and v2:
                    a12 = vargha_delaney_a12(v1, v2)
                    a12_grid[i, j] = a12
                    try:
                        p_val = mannwhitneyu(v1, v2, alternative="two-sided").pvalue
                        ast = "***" if p_val < 0.001 else ("**" if p_val < 0.01 else ("*" if p_val < 0.05 else "ns"))
                    except Exception:
                        ast = ""
                    row_text.append(f"{a12:.2f} {ast}")
                else:
                    a12_grid[i, j] = 0.5
                    row_text.append("N/A")
        text_grid.append(row_text)

    fig_a12 = go.Figure(data=go.Heatmap(
        z=a12_grid, x=all_solvers, y=all_solvers,
        text=text_grid, texttemplate="%{text}", textfont=dict(size=11),
        colorscale="RdYlGn", zmin=0.0, zmax=1.0,
        colorbar=dict(title="<b>A12 (Row < Col)</b><br>Green = Row Superior")
    ))
    apply_publication_theme(
        fig_a12,
        title=f"Figure E: Global Pairwise Effect Size Matrix (Vargha-Delaney A12) — {dim}D",
        width=880, height=620, top_margin=90
    )
    fig_a12.update_xaxes(tickangle=-20)
    out_p = dim_dir(dim) / "a12_effect_size_heatmap.png"
    fig_a12.write_image(str(out_p), scale=2)
    print(f"  ✅ Figure E Exported -> {out_p}")


  ✅ Figure E Exported -> a12_effect_size_heatmap.png


## 📑 Figure F (Appendix): Per-Problem Convergence Curves & Target ECDFs

- 🎯 **Research Question Answered:** What does the exact empirical iteration-by-iteration optimization trajectory look like for each individual test function?
- 💡 **Why We Plot This Data:** Supplementary thesis material. Provides full transparency into search dynamics: median error decay curves $\pm 1\sigma$ alongside the empirical cumulative distribution of target hits ($10^{-8} \dots 10^2$). **Unifies all classical and evolved algorithms into a single 2-panel chart without split folders.**
- 🔍 **Visual Guide:**
  - **Panel 1 (Left):** Log-log plot of evaluations vs. best objective value.
  - **Panel 2 (Right):** Target precision (reversed log scale) vs. fraction of runs reaching the target.


In [54]:
# ── Figure F: Per-Problem Convergence & ECDF Curves (Appendix) ─────────────────
eval_grid = np.logspace(0, 5, 200)
targets = np.logspace(-8, 2, 100)
exported_appendix_figures = 0

conditions_set = sorted(list(set((k[0], k[1]) for k in all_benchmark_data.keys())))
for (dim, noise_std) in conditions_set:
    cond_dir = FIGURES_DIR / f'{dim}D' / f'std_{noise_std}'
    cond_dir.mkdir(parents=True, exist_ok=True)
    for p_id in PROBLEM_IDS:
        key = (dim, noise_std, p_id)
        if key not in all_benchmark_data: continue
        algo_runs = all_benchmark_data[key]
        p_name = BBOB_NAMES.get(p_id, f'f{p_id}')
        p_class = BBOB_CLASSES.get(p_id, 'Unknown')
        
        fig_p = make_subplots(
            rows=1, cols=2,
            subplot_titles=(
                '<b>(A) Empirical Convergence Trajectory</b>',
                '<b>(B) Target Precision Hit Rate (ECDF)</b>'
            ),
            horizontal_spacing=0.14
        )
        for solver, runs in algo_runs.items():
            if not runs: continue
            col = SOLVER_COLORS.get(solver, '#7f7f7f')
            interpolated = [np.interp(eval_grid, evals, raw_vals, left=raw_vals[0], right=raw_vals[-1]) for evals, raw_vals in runs if len(evals) > 0]
            if interpolated:
                fig_p.add_trace(go.Scatter(x=eval_grid, y=np.median(np.array(interpolated), axis=0), mode='lines', name=solver, line=dict(color=col, width=2.2)), row=1, col=1)
            terminals = [r[1][-1] for r in runs if len(r[1]) > 0]
            if terminals:
                fig_p.add_trace(go.Scatter(x=targets, y=[np.mean(np.array(terminals) <= t) for t in targets], mode='lines', name=solver, line=dict(color=col, width=2.2), showlegend=False), row=1, col=2)
                
        super_title = f'BBOB f{p_id}: {p_name} — {dim}D (Noise: σ = {noise_std}, Class: {p_class})'
        fig_p.update_layout(
            template='plotly_white',
            title=dict(
                text=f'<b>{super_title}</b>',
                x=0.03,
                y=0.98,
                font=dict(size=14, color='#2c3e50', family='Inter, Helvetica, Arial, sans-serif')
            ),
            margin=dict(l=65, r=40, t=125, b=60),
            width=1000, height=480,
            legend=dict(
                orientation='h',
                yanchor='bottom',
                y=1.10,
                xanchor='center',
                x=0.5,
                bgcolor='rgba(255,255,255,0.92)',
                bordercolor='rgba(0,0,0,0.12)',
                borderwidth=1,
                font=dict(size=11)
            )
        )
        fig_p.update_xaxes(type='log', title='<b>Evaluations</b>', showgrid=True, gridwidth=1, gridcolor='#EAEAEA', row=1, col=1)
        fig_p.update_yaxes(type='log', title='<b>Best Fitness Value (Δy)</b>', showgrid=True, gridwidth=1, gridcolor='#EAEAEA', row=1, col=1)
        fig_p.update_xaxes(type='log', title='<b>Target Precision (τ)</b>', autorange='reversed', showgrid=True, gridwidth=1, gridcolor='#EAEAEA', row=1, col=2)
        fig_p.update_yaxes(title='<b>Fraction of Runs Solved</b>', range=[-0.05, 1.05], showgrid=True, gridwidth=1, gridcolor='#EAEAEA', row=1, col=2)
        
        out_p_png = cond_dir / f'f{p_id}_all_solvers.png'
        fig_p.write_image(str(out_p_png), scale=2)
        exported_appendix_figures += 1

print(f'  ✅ Exported {exported_appendix_figures} per-problem appendix figures (Figure F)')


  ✅ Exported 30 per-problem appendix figures (Figure F)


# 🎓 Part II: Thesis Visual Storyboard (A → B → C → D Narrative Chain)

The following four figures form a cohesive, rigorous storytelling chain designed specifically for the thesis narrative:
- **Thesis Figure 1 (Did it succeed?):** Convergence Success Rate (Δy < 1e-8) across BBOB problem landscapes.
- **Thesis Figure 2 (How well did it converge?):** Empirical convergence trajectories with shaded Interquartile Range (IQR, 25th–75th percentile) variance bands.
- **Thesis Figure 3 (Does noise break it?):** Two-panel noise robustness analysis (Panel A: Paired clean vs. noisy success rate drop; Panel B: Landscape fragility matrix).
- **Thesis Figure 4 (Is the ranking consistent?):** Two-panel statistical dominance validation (Panel A: Vargha-Delaney {12}$ effect sizes with Mann-Whitney U test significance; Panel B: Dolan-Moré performance profiles $\rho_s(\tau)$).

All thesis figures are exported directly to  for direct inclusion in thesis chapters.


In [ ]:
# ── THESIS Figure 1: Convergence Success Rate (A: Did it succeed?) ────────────
for dim in all_dims:
    for noise_std in all_noise_stds:
        cond_label = f"Noisy (σ={noise_std})" if noise_std > 0 else "Clean (σ=0.0)"
        fig_succ = go.Figure()
        
        for s in all_solvers:
            succ_pcts = []
            for p_id in PROBLEM_IDS:
                key = (dim, noise_std, p_id)
                runs = all_benchmark_data.get(key, {}).get(s, [])
                if not runs:
                    succ_pcts.append(0.0)
                    continue
                terminals = [r[1][-1] for r in runs if len(r[1]) > 0]
                succ = sum(1 for t in terminals if t < CONVERGENCE_THRESHOLD) / len(terminals) * 100 if terminals else 0.0
                succ_pcts.append(succ)
                
            is_llm = "LLaMEA" in s
            fig_succ.add_trace(
                go.Bar(
                    x=[BBOB_NAMES.get(p, f"f{p}") for p in PROBLEM_IDS],
                    y=succ_pcts,
                    name=s,
                    marker=dict(
                        color=SOLVER_COLORS.get(s, "#7f7f7f"),
                        pattern=dict(shape="/" if not is_llm else "", fgcolor="rgba(255,255,255,0.7)", solidity=0.5)
                    ),
                    opacity=1.0 if is_llm else 0.88
                )
            )
            
        fig_succ.add_hline(
            y=50, line_dash="dash", line_color="rgba(0,0,0,0.3)",
            annotation_text="50% Benchmark Threshold", annotation_position="top right",
            annotation_font=dict(size=10, color="gray")
        )
        
        apply_publication_theme(
            fig_succ,
            title=f"Thesis Figure 1: Convergence Success Rate (Δy < 1e-8) — {dim}D ({cond_label})",
            width=1020, height=530, top_margin=95
        )
        fig_succ.update_layout(barmode="group")
        fig_succ.update_xaxes(categoryorder="array", categoryarray=PROBLEM_ORDER, tickangle=-15)
        fig_succ.update_yaxes(title="<b>Success Rate (%)</b>", range=[0, 115])
        
        out_p = dim_thesis_dir(dim) / f"figure_1_success_rate_std_{noise_std}.png"
        fig_succ.write_image(str(out_p), scale=2)
        print(f"  ✅ Thesis Figure 1 Exported -> {out_p}")


In [ ]:
# ── THESIS Figure 2: Convergence Curves with Shaded IQR Bands (B: Accuracy) ───
eval_grid = np.logspace(0, 5, 200)

for dim in all_dims:
    for noise_std in all_noise_stds:
        fig_traj = make_subplots(
            rows=2, cols=3,
            subplot_titles=[f"<b>{BBOB_NAMES.get(p, f'f{p}')}</b>" for p in PROBLEM_IDS] + [""],
            horizontal_spacing=0.08,
            vertical_spacing=0.18
        )
        for idx, p_id in enumerate(PROBLEM_IDS):
            r_idx = idx // 3 + 1
            c_idx = idx % 3 + 1
            key = (dim, noise_std, p_id)
            if key not in all_benchmark_data: continue
            solvers_data = all_benchmark_data[key]
            
            for s in all_solvers:
                if s not in solvers_data or not solvers_data[s]: continue
                runs = solvers_data[s]
                interpolated = [np.interp(eval_grid, evals, raw_vals, left=raw_vals[0], right=raw_vals[-1]) for evals, raw_vals in runs if len(evals) > 0]
                if not interpolated: continue
                arr = np.array(interpolated)
                med = np.median(arr, axis=0)
                q25 = np.percentile(arr, 25, axis=0)
                q75 = np.percentile(arr, 75, axis=0)
                
                col = SOLVER_COLORS.get(s, "#7f7f7f")
                hex_c = col.lstrip("#")
                rgb = tuple(int(hex_c[i:i+2], 16) for i in (0, 2, 4))
                rgba_fill = f"rgba({rgb[0]}, {rgb[1]}, {rgb[2]}, 0.15)"
                
                fig_traj.add_trace(
                    go.Scatter(
                        x=np.concatenate([eval_grid, eval_grid[::-1]]),
                        y=np.concatenate([q75, q25[::-1]]),
                        fill="toself",
                        fillcolor=rgba_fill,
                        line=dict(color="rgba(255,255,255,0)", width=0),
                        showlegend=False,
                        hoverinfo="skip"
                    ),
                    row=r_idx, col=c_idx
                )
                
                is_llm = "LLaMEA" in s
                fig_traj.add_trace(
                    go.Scatter(
                        x=eval_grid, y=med, mode="lines", name=s,
                        line=dict(
                            color=col,
                            width=2.5 if is_llm else 1.5,
                            dash="solid" if is_llm else "dash"
                        ),
                        legend="legend1",
                        showlegend=(idx == 0)
                    ),
                    row=r_idx, col=c_idx
                )
                fig_traj.update_xaxes(type="log", title="Evaluations" if r_idx==2 else None, row=r_idx, col=c_idx)
                fig_traj.update_yaxes(type="log", title="Best Δy" if c_idx==1 else None, row=r_idx, col=c_idx)
                
        cond_title = f"Noisy (σ={noise_std})" if noise_std > 0 else "Clean (σ=0.0)"
        apply_publication_theme(
            fig_traj,
            title=f"Thesis Figure 2: Empirical Convergence Trajectories with IQR (25th–75th percentile) — {dim}D ({cond_title})",
            width=1240, height=740, top_margin=130
        )
        fig_traj.update_layout(
            legend=dict(
                orientation="h",
                yanchor="bottom", y=1.08,
                xanchor="center", x=0.5,
                bgcolor="rgba(255,255,255,0.95)",
                bordercolor="rgba(0,0,0,0.12)",
                borderwidth=1,
                font=dict(size=11)
            )
        )
        out_p = dim_thesis_dir(dim) / f"figure_2_convergence_std_{noise_std}.png"
        fig_traj.write_image(str(out_p), scale=2)
        print(f"  ✅ Thesis Figure 2 Exported -> {out_p}")


In [ ]:
# ── THESIS Figure 3: Noise Robustness Profile (C: Does noise break it?) ───────
clean_std = 0.0
noisy_stds = [s for s in all_noise_stds if s > 0]
noisy_std = noisy_stds[0] if noisy_stds else 0.05

for dim in all_dims:
    fig_robust = make_subplots(
        rows=1, cols=2,
        subplot_titles=(
            "<b>(A) Success Rate Drop under Noise (Clean vs. Noisy)</b>",
            f"<b>(B) Landscape Fragility Index Matrix (σ={noisy_std})</b>"
        ),
        column_widths=[0.45, 0.55],
        horizontal_spacing=0.14
    )
    
    solvers_clean_succ = []
    solvers_noisy_succ = []
    
    for s in all_solvers:
        c_terms, n_terms = [], []
        for p_id in PROBLEM_IDS:
            c_runs = all_benchmark_data.get((dim, clean_std, p_id), {}).get(s, [])
            n_runs = all_benchmark_data.get((dim, noisy_std, p_id), {}).get(s, [])
            c_terms.extend([r[1][-1] for r in c_runs if len(r[1]) > 0])
            n_terms.extend([r[1][-1] for r in n_runs if len(r[1]) > 0])
        c_rate = (sum(1 for t in c_terms if t < CONVERGENCE_THRESHOLD) / len(c_terms) * 100) if c_terms else 0.0
        n_rate = (sum(1 for t in n_terms if t < CONVERGENCE_THRESHOLD) / len(n_terms) * 100) if n_terms else 0.0
        solvers_clean_succ.append(c_rate)
        solvers_noisy_succ.append(n_rate)
        
    for i, s in enumerate(all_solvers):
        c_val = solvers_clean_succ[i]
        n_val = solvers_noisy_succ[i]
        col = SOLVER_COLORS.get(s, "#7f7f7f")
        fig_robust.add_trace(
            go.Scatter(
                x=[s, s], y=[c_val, n_val], mode="lines",
                line=dict(color="#888888", width=2, dash="dot"),
                showlegend=False, hoverinfo="skip"
            ), row=1, col=1
        )
        fig_robust.add_trace(
            go.Scatter(
                x=[s], y=[c_val], mode="markers",
                marker=dict(size=11, color=col, symbol="circle-open", line=dict(width=2.5, color=col)),
                name=f"Clean (σ={clean_std})" if i==0 else s,
                legend="legend1",
                showlegend=(i==0)
            ), row=1, col=1
        )
        fig_robust.add_trace(
            go.Scatter(
                x=[s], y=[n_val], mode="markers",
                marker=dict(size=11, color=col, symbol="circle"),
                name=f"Noisy (σ={noisy_std})" if i==0 else s,
                legend="legend1",
                showlegend=(i==0)
            ), row=1, col=1
        )
        
    fig_robust.update_yaxes(title="<b>Success Rate (%)</b>", range=[-5, 105], row=1, col=1)
    fig_robust.update_xaxes(tickangle=-25, row=1, col=1)
    
    deg_grid = np.zeros((len(PROBLEM_IDS), len(all_solvers)))
    deg_text = []
    for i, p_id in enumerate(PROBLEM_IDS):
        row_t = []
        for j, s in enumerate(all_solvers):
            c_runs = all_benchmark_data.get((dim, clean_std, p_id), {}).get(s, [])
            n_runs = all_benchmark_data.get((dim, noisy_std, p_id), {}).get(s, [])
            c_terms = [r[1][-1] for r in c_runs if len(r[1]) > 0]
            n_terms = [r[1][-1] for r in n_runs if len(r[1]) > 0]
            if c_terms and n_terms:
                c_med, n_med = np.median(c_terms), np.median(n_terms)
                if c_med == 0.0 and n_med == 0.0:
                    deg = 0.0
                    row_t.append("0.0")
                elif c_med == 0.0 or n_med == 0.0:
                    deg = np.nan
                    row_t.append("—")
                else:
                    deg = np.log10(n_med) - np.log10(c_med)
                    deg = float(np.clip(deg, -5.0, 5.0))
                    row_t.append(f"{deg:+.1f}")
                deg_grid[i, j] = deg
            else:
                deg_grid[i, j] = np.nan
                row_t.append("N/A")
        deg_text.append(row_t)
        
    fig_robust.add_trace(
        go.Heatmap(
            z=deg_grid, x=all_solvers, y=[BBOB_NAMES.get(p, f"f{p}") for p in PROBLEM_IDS],
            text=deg_text, texttemplate="%{text}", textfont=dict(size=10),
            colorscale="Plasma", zmid=0.0, zmin=-5.0, zmax=5.0,
            colorbar=dict(title="<b>Δlog10 Error</b>", x=1.02, len=0.85),
            showscale=True
        ), row=1, col=2
    )
    fig_robust.update_xaxes(tickangle=-25, row=1, col=2)
    fig_robust.update_yaxes(categoryorder="array", categoryarray=[BBOB_NAMES.get(p, f"f{p}") for p in PROBLEM_IDS], autorange="reversed", row=1, col=2)
    
    apply_publication_theme(
        fig_robust,
        title=f"Thesis Figure 3: Empirical Noise Robustness & Landscape Degradation Profile — {dim}D",
        width=1220, height=530, top_margin=95
    )
    
    out_p = dim_thesis_dir(dim) / "figure_3_noise_robustness.png"
    fig_robust.write_image(str(out_p), scale=2)
    print(f"  ✅ Thesis Figure 3 Exported -> {out_p}")


In [ ]:
# ── THESIS Figure 4: Statistical Validation & Performance Profiles (D: Ranking) 
for dim in all_dims:
    fig_stat = make_subplots(
        rows=1, cols=2,
        subplot_titles=(
            f"<b>(A) Pairwise Vargha-Delaney (A12) Effect Size — {dim}D</b>",
            f"<b>(B) Dolan-Moré Performance Profiles ρ(τ) (Noisy σ={noisy_std}) — {dim}D</b>"
        ),
        column_widths=[0.48, 0.52],
        horizontal_spacing=0.18
    )
    
    # Panel A: A12 Heatmap
    def vargha_delaney_a12(s1, s2):
        m, n = len(s1), len(s2)
        if m == 0 or n == 0: return 0.5
        r1 = np.sum([np.sum(x < s2) + 0.5 * np.sum(x == s2) for x in s1])
        return float(r1 / (m * n))
        
    solver_res = {s: [] for s in all_solvers}
    for (d_val, n_std, p_id), s_dict in all_benchmark_data.items():
        if d_val != dim: continue
        for s in all_solvers:
            terms = [r[1][-1] for r in s_dict.get(s, []) if len(r[1]) > 0]
            solver_res[s].extend(terms)
            
    a12_grid = np.zeros((len(all_solvers), len(all_solvers)))
    text_grid = []
    for i, s1 in enumerate(all_solvers):
        row_t = []
        for j, s2 in enumerate(all_solvers):
            if i == j:
                a12_grid[i, j] = 0.5
                row_t.append("—")
            else:
                v1, v2 = solver_res[s1], solver_res[s2]
                if v1 and v2:
                    a12 = vargha_delaney_a12(v1, v2)
                    a12_grid[i, j] = a12
                    try:
                        p_val = mannwhitneyu(v1, v2, alternative="two-sided").pvalue
                        ast = "***" if p_val < 0.001 else ("**" if p_val < 0.01 else ("*" if p_val < 0.05 else "ns"))
                    except Exception: ast = ""
                    row_t.append(f"{a12:.2f} {ast}")
                else:
                    a12_grid[i, j] = 0.5
                    row_t.append("N/A")
        text_grid.append(row_t)
        
    fig_stat.add_trace(
        go.Heatmap(
            z=a12_grid, x=all_solvers, y=all_solvers,
            text=text_grid, texttemplate="%{text}", textfont=dict(size=9.5),
            colorscale="RdYlGn", zmin=0.0, zmax=1.0,
            colorbar=dict(title="<b>A12</b>", x=0.42, len=0.85),
            showscale=True
        ), row=1, col=1
    )
    fig_stat.update_xaxes(tickangle=-25, row=1, col=1)
    
    # Panel B: Dolan-Moré profiles for noisy condition
    cond_keys = [k for k in all_benchmark_data.keys() if k[0] == dim and k[1] == noisy_std]
    perf_matrix = {s: {} for s in all_solvers}
    for p_key in cond_keys:
        for s in all_solvers:
            terms = [r[1][-1] for r in all_benchmark_data[p_key].get(s, []) if len(r[1]) > 0]
            perf_matrix[s][p_key] = np.median(terms) if terms else 1e9
    best_perf = {p_key: min([perf_matrix[s][p_key] for s in all_solvers]) + 1e-12 for p_key in cond_keys}
    tau_grid = np.logspace(0, 4, 150)
    
    for s in all_solvers:
        d_curve = []
        for tau in tau_grid:
            solved = sum(1 for p_key in cond_keys if (perf_matrix[s][p_key] + 1e-12) / best_perf[p_key] <= tau)
            d_curve.append(solved / max(1, len(cond_keys)))
        is_llm = "LLaMEA" in s
        fig_stat.add_trace(
            go.Scatter(
                x=tau_grid, y=d_curve, mode="lines", name=s,
                line=dict(
                    color=SOLVER_COLORS.get(s, "#7f7f7f"),
                    width=2.5 if is_llm else 1.5,
                    dash="solid" if is_llm else "dash"
                ),
                legend="legend2",
                showlegend=True
            ), row=1, col=2
        )
    fig_stat.update_xaxes(type="log", title="<b>Performance Ratio Factor (τ)</b>", row=1, col=2)
    fig_stat.update_yaxes(title="<b>Fraction Solved (ρ(τ))</b>", range=[-0.02, 1.02], row=1, col=2)
    
    apply_publication_theme(
        fig_stat,
        title=f"Thesis Figure 4: Statistical Dominance & Benchmark Ranking Validation — {dim}D",
        width=1280, height=550, top_margin=95
    )
    
    out_p = dim_thesis_dir(dim) / "figure_4_ranking_and_statistical_validation.png"
    fig_stat.write_image(str(out_p), scale=2)
    print(f"  ✅ Thesis Figure 4 Exported -> {out_p}")
